# Notebook 3: Exploratory Data Analysis (EDA)
## Car Price Prediction Project

**Objective:** Understand distributions, relationships, correlations, and outliers in the cleaned dataset.

---

## 3.1 Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

df = pd.read_csv('/content/drive/MyDrive/car_price_cleaned.csv')
print(f"Loaded: {df.shape}")
df.head()

## 3.2 Target Variable Distribution (Price)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Histogram
axes[0].hist(df['price'], bins=30, color='steelblue', edgecolor='white')
axes[0].set_title('Price Distribution')
axes[0].set_xlabel('Price')
axes[0].set_ylabel('Frequency')

# Boxplot
axes[1].boxplot(df['price'], vert=True)
axes[1].set_title('Price Boxplot')
axes[1].set_ylabel('Price')

# Log-transformed distribution
axes[2].hist(np.log1p(df['price']), bins=30, color='coral', edgecolor='white')
axes[2].set_title('Log(Price) Distribution')
axes[2].set_xlabel('Log(Price)')
axes[2].set_ylabel('Frequency')

plt.tight_layout()
plt.show()

print(f"Skewness of price: {df['price'].skew():.3f}")
print(f"Skewness of log(price): {np.log1p(df['price']).skew():.3f}")

## 3.3 Numerical Features Distributions

In [ ]:
numerical_cols = df.select_dtypes(include=[np.number]).columns.tolist()
numerical_cols.remove('price')  # exclude target
print(f"Numerical features: {len(numerical_cols)}")
print(numerical_cols)

In [ ]:
# Histograms of all numerical features
n_cols = 4
n_rows = (len(numerical_cols) + n_cols - 1) // n_cols
fig, axes = plt.subplots(n_rows, n_cols, figsize=(20, n_rows * 4))
axes = axes.flatten()

for i, col in enumerate(numerical_cols):
    axes[i].hist(df[col], bins=25, color='steelblue', edgecolor='white')
    axes[i].set_title(f'{col} (skew={df[col].skew():.2f})')

for j in range(i+1, len(axes)):
    axes[j].set_visible(False)

plt.tight_layout()
plt.show()

## 3.4 Skewness Summary

In [ ]:
skew_df = df[numerical_cols + ['price']].skew().sort_values(ascending=False)
skew_df = pd.DataFrame({'Feature': skew_df.index, 'Skewness': skew_df.values})
skew_df['Status'] = skew_df['Skewness'].apply(
    lambda x: 'Highly Skewed' if abs(x) > 1 else ('Moderately Skewed' if abs(x) > 0.5 else 'OK')
)
print("Skewness of all numerical features:")
skew_df

## 3.5 Categorical Features

In [ ]:
cat_cols = df.select_dtypes(include='object').columns.tolist()
print(f"Categorical features: {len(cat_cols)}")
print(cat_cols)

In [ ]:
# Bar plots for categorical features
n_cols_plot = 3
n_rows_plot = (len(cat_cols) + n_cols_plot - 1) // n_cols_plot
fig, axes = plt.subplots(n_rows_plot, n_cols_plot, figsize=(18, n_rows_plot * 4))
axes = axes.flatten()

for i, col in enumerate(cat_cols):
    df[col].value_counts().plot(kind='bar', ax=axes[i], color='steelblue', edgecolor='white')
    axes[i].set_title(col)
    axes[i].tick_params(axis='x', rotation=45)

for j in range(i+1, len(axes)):
    axes[j].set_visible(False)

plt.tight_layout()
plt.show()

## 3.6 Price by Categorical Features

In [ ]:
# Price distribution by brand
brand_avg = df.groupby('brand')['price'].mean().sort_values(ascending=False)

plt.figure(figsize=(14, 6))
brand_avg.plot(kind='bar', color='steelblue', edgecolor='white')
plt.title('Average Price by Brand')
plt.ylabel('Average Price ($)')
plt.xlabel('Brand')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

In [ ]:
# Price boxplots by key categorical features
key_cats = ['fueltype', 'aspiration', 'carbody', 'drivewheel', 'enginetype']
fig, axes = plt.subplots(1, len(key_cats), figsize=(22, 5))

for i, col in enumerate(key_cats):
    df.boxplot(column='price', by=col, ax=axes[i])
    axes[i].set_title(f'Price by {col}')
    axes[i].set_xlabel(col)
    axes[i].tick_params(axis='x', rotation=45)

plt.suptitle('')
plt.tight_layout()
plt.show()

## 3.7 Correlation Analysis

In [ ]:
# Correlation heatmap
corr = df[numerical_cols + ['price']].corr()

plt.figure(figsize=(16, 12))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='RdBu_r',
            center=0, vmin=-1, vmax=1, linewidths=0.5)
plt.title('Correlation Heatmap')
plt.tight_layout()
plt.show()

In [ ]:
# Top correlations with price
price_corr = corr['price'].drop('price').sort_values(ascending=False)
print("Correlation with Price (sorted):")
print(price_corr.to_string())

## 3.8 Scatter Plots — Top Correlated Features vs Price

In [ ]:
top_features = price_corr.abs().sort_values(ascending=False).head(6).index.tolist()

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.flatten()

for i, col in enumerate(top_features):
    axes[i].scatter(df[col], df['price'], alpha=0.5, color='steelblue', s=30)
    axes[i].set_xlabel(col)
    axes[i].set_ylabel('Price')
    axes[i].set_title(f'{col} vs Price (r={price_corr[col]:.2f})')

plt.tight_layout()
plt.show()

## 3.9 Highly Correlated Feature Pairs (Multicollinearity)

In [ ]:
# Find feature pairs with correlation > 0.8
high_corr_pairs = []
for i in range(len(corr.columns)):
    for j in range(i+1, len(corr.columns)):
        if abs(corr.iloc[i, j]) > 0.8 and corr.columns[i] != 'price' and corr.columns[j] != 'price':
            high_corr_pairs.append({
                'Feature 1': corr.columns[i],
                'Feature 2': corr.columns[j],
                'Correlation': corr.iloc[i, j]
            })

high_corr_df = pd.DataFrame(high_corr_pairs).sort_values('Correlation', ascending=False)
print("Highly correlated feature pairs (|r| > 0.8):")
high_corr_df

## 3.10 Outlier Detection

In [ ]:
# Boxplots for outlier detection in key numerical features
outlier_cols = ['price', 'horsepower', 'enginesize', 'curbweight', 'citympg', 'highwaympg']

fig, axes = plt.subplots(2, 3, figsize=(18, 8))
axes = axes.flatten()

for i, col in enumerate(outlier_cols):
    axes[i].boxplot(df[col])
    axes[i].set_title(f'{col}')

plt.suptitle('Outlier Detection via Boxplots', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# Count outliers using IQR method
def count_outliers(series):
    Q1 = series.quantile(0.25)
    Q3 = series.quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    return ((series < lower) | (series > upper)).sum()

print("Outlier counts (IQR method):")
for col in numerical_cols + ['price']:
    n = count_outliers(df[col])
    if n > 0:
        print(f"  {col}: {n} outliers")

## 3.11 Pairplot of Top Features

In [ ]:
top4 = price_corr.abs().sort_values(ascending=False).head(4).index.tolist()
sns.pairplot(df[top4 + ['price']], diag_kind='kde', plot_kws={'alpha': 0.5})
plt.suptitle('Pairplot of Top Correlated Features with Price', y=1.02)
plt.show()

---
## EDA Key Findings

1. **Price is right-skewed** (skewness ~1.78) → log transformation recommended
2. **Top correlated features** with price: `enginesize`, `curbweight`, `horsepower`, `carwidth`
3. **Highly correlated pairs** (multicollinearity): `citympg` & `highwaympg`, `carlength` & `wheelbase`
4. **Brand has strong influence** on price — luxury brands (BMW, Porsche, Jaguar) have much higher average prices
5. **Several features are skewed** and may benefit from log transformation
6. **Outliers exist** in price, horsepower, and enginesize

**Next step →** Notebook 04: Feature Engineering